# Line Predictors
### Notebook initialisation

In [ ]:
print(__debug__)

import logging
logging.basicConfig(level=logging.INFO)

import torch, random, os, cv2
import numpy as np

seed = 1

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
cv2.setRNGSeed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
os.environ["PYTHONHASHSEED"] = str(seed)

import seaborn as sns
sns.set_theme(style="whitegrid", context="paper")

### Dataset Loading

In [ ]:
robot_data_folder_location = "../example_datasets/example_small_aruco1"
vrs_file_location = "../example_datasets/small_aruco1_sitting_20fps.vrs"

from pose_estimation.headset_data import HeadsetData, create_robot_bound_headset_data
from pose_estimation.robot_environment import RobotEnvironment, GatheredRobotData, visualize_robot_camera_environment_combo, XYZImageGenerationConfig, ICPAlignmentConfig

robot_data = GatheredRobotData.from_folder(robot_data_folder_location)
robot_env = RobotEnvironment.from_gathered_robot_data(
        robot_data = robot_data,
        number_of_sampled_datapoints=10,
        sample_datapoints_based_on_aruco_corectness = False,
        only_sample_robot_datapoints_w_marker_estimates = True,
        markers_use_advanced_removal=True,
        est3d_xyz_image_gen_config = XYZImageGenerationConfig(iforest_contamination=0.05, use_depth_images_if_provided=True),
        est3d_xyz_icp_config=ICPAlignmentConfig(do_alginment=False)
)

labeled_headset_data = create_robot_bound_headset_data(
        headset_data = HeadsetData.from_vrs_file(vrs_file_location),
        robot_data = robot_data
)

visualize_loaded_data = False

if visualize_loaded_data:
    visualize_robot_camera_environment_combo(robot_env=robot_env, headset_data=labeled_headset_data)

## Hyperparameters
### Simple Predictor

In [ ]:
from pose_estimation.pose_pred_points import OnlyPointsPredictor, ExtractAndMatchWrapperConfig
from pose_estimation.pose_pred_points_lines import LinePredictor, LineGenerator, PnPLOptimizerConfig
from pose_estimation.predictor_grader import PredictionOnDataset

default_point_predictor = LinePredictor(
        cam2_intrinsic_mtx=labeled_headset_data.intrinsic_cam_mtx,
        cam1_bgr_images=robot_env.robot_bgr_images,
        cam1_xyz_images=robot_env.robot_xyz_images,
        extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(),
        cam1_line_generator=LineGenerator(
            visualize_cleanup=False
        ),
        debug_visualize_pnpl=False
)

PredictionOnDataset(predictor = default_point_predictor,headset_data = labeled_headset_data).print_summary()

### Line Relevance

In [ ]:
from pose_estimation.predictor_grader import *
from pose_estimation.pose_pred_points_lines import *

line_relevancies = [
    GradablePosePredictor(
        creator=LinePredictor.get_creation_function(
                cam2_intrinsic_mtx=labeled_headset_data.intrinsic_cam_mtx,
                extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
                    ransac_config=pose_estimation_ransaac_config_10ms
                ),
                pnpl_optimisation_conf=PnPLOptimizerConfig(
                    line_relevance=line_relevance
                )
            ),
        category = "Points & Lines:",
        name=f"LR: {line_relevance:.2f}",
        value=line_relevance
    )
    for line_relevance in np.arange(0, 0.2, 0.02)
]

Now those can be used to create a grader object for multiple `PosePredictor` variants.

In [ ]:
line_relevance_grader = NPredictors1DatasetGrader(
    gradable_pose_predictors=line_relevancies,
    headset_data = labeled_headset_data,
    robot_env = robot_env,
)

In [ ]:
fig, axes1 = plt.subplots(2,1, figsize = (16, 12))
line_relevance_grader.plot_value_vs_error(
    ax = axes1[0],
    x_axis_label= "Line relevancy",
    x_axis_title_name = "Line importance",
    error_type = SingleValueErrorType.AVG_TRANSLATIONAL,
    use_category=True
)

line_relevance_grader.plot_value_vs_error(
    ax = axes1[1],
    x_axis_label= "Line relevancy",
    x_axis_title_name = "Line importance",
    error_type = SingleValueErrorType.AVG_ROTATIONAL,
    use_category=True
)
plt.show()